# SLAE-CS on a Colab GPU

Installs SLAE-CS, checks the installation against the reference output of the paper's environment, and
predicts backbone amide ¹⁵N chemical shifts (SLAE-CS, 10-seed ensemble) and their expected error
(SLAE-ERROR) for a protein structure.

**Runtime → Change runtime type → GPU** (a T4 is enough). The install takes a few minutes.

Code: [WaymentSteeleLab/SLAE-CS](https://github.com/WaymentSteeleLab/SLAE-CS) · weights:
[hws-lab/SLAE-CS](https://huggingface.co/hws-lab/SLAE-CS)

## 1. Code and environment

*While the repositories are private:* add two Colab secrets (key icon in the left panel, *Notebook access* on):
`GH_TOKEN`, a GitHub token that can read the code repository, and `HF_TOKEN`, a Hugging Face token that can
read `hws-lab/SLAE-CS`.

In [ ]:
!nvidia-smi -L

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    for k in ("GH_TOKEN", "HF_TOKEN"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

REPO = "github.com/WaymentSteeleLab/SLAE-CS.git"
if not os.path.isdir("SLAE-CS"):
    url = f"https://{os.environ['GH_TOKEN']}@{REPO}" if os.environ.get("GH_TOKEN") else f"https://{REPO}"
    subprocess.run(["git", "clone", "-q", url, "SLAE-CS"], check=True)
%cd SLAE-CS
!git log -1 --format='%h %s'

In [ ]:
# The versions of the paper's environment (containers/slae/pip_list.txt), CUDA 12.4 builds.
# pip may warn about packages preinstalled in Colab (e.g. torchaudio); they are not used.
# python-dotenv: needed by atomworks 1.0.3 but not declared by it.
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q torch-scatter==2.1.2 torch-cluster==1.6.3 torch-sparse==0.6.18 -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install -q torch==2.6.0 torch-geometric==2.7.0 e3nn==0.5.8 einops==0.8.1 einx==0.3.0 esm==3.2.1 \
    lightning==2.5.6 hydra-core==1.3.2 omegaconf==2.3.0 loguru==0.7.3 opt-einsum-fx==0.1.4 \
    torch-runstats==0.2.0 biotite==1.5.0 atomworks==1.0.3 python-dotenv==1.2.1 safetensors==0.6.2 \
    huggingface-hub==0.36.0 numpy==1.26.4
!python -c "import torch, torch_geometric; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| PyG', torch_geometric.__version__)"

## 2. Weights and installation test

In [ ]:
!python scripts/download_from_hf.py --no-data        # weights only, into model/

In [ ]:
# predict.py on the three structures in examples/, compared with the reference output (NVIDIA A100)
!python tests/smoke_test.py --device cuda

## 3. Your own structure

One protein per PDB file. Upload a file, or keep `PDB` pointing at one of the examples. Only the first model
of a multi-model file is used.

In [ ]:
PDB = "examples/structures/15419_1.pdb"
try:
    from google.colab import files
    up = files.upload()                      # cancel to keep the example
    if up:
        PDB = next(iter(up))
except Exception:
    pass
print("structure:", PDB)
!python predict.py --weights model/SLAE-CS-PANAV --error-model model/SLAE-ERROR --pdb "$PDB" --out prediction --device cuda

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("prediction.csv")
print(f"{df.protein.nunique()} protein(s), {len(df)} residues (prolines have no amide N and are left out)")
display(df[["protein", "chain", "resnum", "resname", "N_mean", "N_std", "N_abs_error"]].head(10))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
a1.errorbar(df.resnum, df.N_mean, yerr=df.N_std, fmt="o", ms=3, lw=0.8, capsize=0)
a1.invert_yaxis()
a1.set_ylabel("predicted 15N (ppm)")
a1.set_title("SLAE-CS: mean of 10 seeds (bars: spread between seeds)")
a2.bar(df.resnum, df.N_abs_error, width=0.8)
a2.set_ylabel("expected |error| (ppm)")
a2.set_xlabel("residue number")
a2.set_title("SLAE-ERROR")
plt.tight_layout()
plt.show()

In [ ]:
try:
    from google.colab import files
    files.download("prediction.csv")
    files.download("prediction.json")
except Exception:
    print("prediction.csv and prediction.json are in", os.getcwd())